In [ ]:
# -*- coding: utf-8 -*-
"""
BigAlphaMixer + Fourier Dual-Domain BigAlphaMixer 融合提交。

平台公榜：
    加载两个已经训练完成的JSON模型进行推理。

平台私榜：
    调用fusion_train.train_and_save，从零训练两个模型。

融合方式：
    每个交易日分别对v1和v2预测做横截面标准化，
    然后按50% + 50%融合。

输出：
    date、instrument、score
"""

from __future__ import annotations

import json
from pathlib import Path

import dai
import numpy as np
import pandas as pd
import structlog
import torch

from torch import nn

class MultiScaleTemporalBlock(nn.Module):

    def __init__(self, d_model: int, kernels: tuple[int, ...]=(3, 5, 9), dropout: float=0.1) -> None:
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel, padding=kernel // 2, groups=d_model), nn.GELU(), nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=1)) for kernel in kernels])
        self.merge = nn.Conv1d(in_channels=d_model * len(kernels), out_channels=d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x_channels = x.transpose(1, 2)
        branch_outputs = [branch(x_channels) for branch in self.branches]
        mixed = torch.cat(branch_outputs, dim=1)
        mixed = self.merge(mixed)
        mixed = mixed.transpose(1, 2)
        return self.norm(residual + self.dropout(mixed))

class AttentionPooling(nn.Module):

    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = self.attention(x)
        weights = torch.softmax(weights, dim=1)
        return torch.sum(x * weights, dim=1)

class BigAlphaMixer(nn.Module):

    def __init__(self, input_dim: int=25, seq_len: int=48, d_model: int=128, group_dim: int=32, nhead: int=8, num_layers: int=2, dim_feedforward: int=256, temporal_layers: int=2, dropout: float=0.1) -> None:
        super().__init__()
        if input_dim != 25:
            raise ValueError(f'BigAlphaMixer要求25个输入字段，实际为{input_dim}')
        self.price_encoder = nn.Sequential(nn.Linear(10, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.trade_encoder = nn.Sequential(nn.Linear(3, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.book_volume_encoder = nn.Sequential(nn.Linear(6, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.order_count_encoder = nn.Sequential(nn.Linear(6, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.group_fusion = nn.Sequential(nn.Linear(group_dim * 4, d_model), nn.GELU(), nn.LayerNorm(d_model))
        self.position_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        self.temporal_blocks = nn.ModuleList([MultiScaleTemporalBlock(d_model=d_model, kernels=(3, 5, 9), dropout=dropout) for _ in range(temporal_layers)])
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.global_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fusion_gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.fusion_norm = nn.LayerNorm(d_model)
        self.pooling = AttentionPooling(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def encode_fields(self, x: torch.Tensor) -> torch.Tensor:
        price = self.price_encoder(x[..., 0:10])
        trade = self.trade_encoder(x[..., 10:13])
        book_volume = self.book_volume_encoder(x[..., 13:19])
        order_count = self.order_count_encoder(x[..., 19:25])
        grouped = torch.cat([price, trade, book_volume, order_count], dim=-1)
        return self.group_fusion(grouped)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.encode_fields(x)
        tokens = tokens + self.position_embedding
        local_features = tokens
        for block in self.temporal_blocks:
            local_features = block(local_features)
        global_features = self.global_encoder(tokens)
        gate = self.fusion_gate(torch.cat([local_features, global_features], dim=-1))
        fused = gate * global_features + (1.0 - gate) * local_features
        fused = self.fusion_norm(fused)
        pooled = self.pooling(fused)
        return self.head(pooled).squeeze(-1)

class FD_MultiScaleTemporalBlock(nn.Module):

    def __init__(self, d_model: int, kernels: tuple[int, ...]=(3, 5, 9), dropout: float=0.1) -> None:
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv1d(d_model, d_model, kernel_size=kernel, padding=kernel // 2, groups=d_model), nn.GELU(), nn.Conv1d(d_model, d_model, kernel_size=1)) for kernel in kernels])
        self.merge = nn.Conv1d(d_model * len(kernels), d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        channel_first = x.transpose(1, 2)
        branches = [branch(channel_first) for branch in self.branches]
        mixed = torch.cat(branches, dim=1)
        mixed = self.merge(mixed)
        mixed = mixed.transpose(1, 2)
        return self.norm(residual + self.dropout(mixed))

class FD_AttentionPooling(nn.Module):

    def __init__(self, d_model: int) -> None:
        super().__init__()
        hidden = max(d_model // 2, 16)
        self.attention = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.attention(x), dim=1)
        return torch.sum(x * weights, dim=1)

class FD_GroupedFieldEncoder(nn.Module):

    def __init__(self, group_dim: int, d_model: int) -> None:
        super().__init__()
        self.price_encoder = nn.Sequential(nn.Linear(10, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.trade_encoder = nn.Sequential(nn.Linear(3, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.book_volume_encoder = nn.Sequential(nn.Linear(6, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.order_count_encoder = nn.Sequential(nn.Linear(6, group_dim), nn.GELU(), nn.LayerNorm(group_dim))
        self.fusion = nn.Sequential(nn.Linear(group_dim * 4, d_model), nn.GELU(), nn.LayerNorm(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        price = self.price_encoder(x[..., 0:10])
        trade = self.trade_encoder(x[..., 10:13])
        book_volume = self.book_volume_encoder(x[..., 13:19])
        order_count = self.order_count_encoder(x[..., 19:25])
        grouped = torch.cat([price, trade, book_volume, order_count], dim=-1)
        return self.fusion(grouped)

class FourierDualDomainBigAlphaMixer(nn.Module):

    def __init__(self, input_dim: int=25, seq_len: int=240, recent_len: int=48, freq_bins: int=12, d_model: int=128, group_dim: int=32, nhead: int=8, time_transformer_layers: int=2, frequency_transformer_layers: int=1, temporal_layers: int=2, dim_feedforward: int=256, dropout: float=0.1) -> None:
        super().__init__()
        if input_dim != 25:
            raise ValueError(f'模型要求25个输入字段，实际为{input_dim}')
        if recent_len > seq_len:
            raise ValueError('recent_len不能大于seq_len')
        maximum_frequency_bins = seq_len // 2
        if freq_bins > maximum_frequency_bins:
            raise ValueError('freq_bins超过可用频率数量')
        self.input_dim = input_dim
        self.seq_len = seq_len
        self.recent_len = recent_len
        self.freq_bins = freq_bins
        self.field_encoder = FD_GroupedFieldEncoder(group_dim=group_dim, d_model=d_model)
        self.time_position_embedding = nn.Parameter(torch.zeros(1, recent_len, d_model))
        self.temporal_blocks = nn.ModuleList([FD_MultiScaleTemporalBlock(d_model=d_model, kernels=(3, 5, 9), dropout=dropout) for _ in range(temporal_layers)])
        time_encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.time_transformer = nn.TransformerEncoder(time_encoder_layer, num_layers=time_transformer_layers)
        self.time_fusion_gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.time_fusion_norm = nn.LayerNorm(d_model)
        self.time_pooling = FD_AttentionPooling(d_model)
        self.frequency_projection = nn.Sequential(nn.Linear(input_dim * 2, d_model), nn.GELU(), nn.LayerNorm(d_model))
        self.frequency_position_embedding = nn.Parameter(torch.zeros(1, freq_bins, d_model))
        frequency_encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.frequency_transformer = nn.TransformerEncoder(frequency_encoder_layer, num_layers=frequency_transformer_layers)
        self.frequency_pooling = FD_AttentionPooling(d_model)
        self.domain_gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model), nn.Sigmoid())
        self.domain_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.time_position_embedding, std=0.02)
        nn.init.trunc_normal_(self.frequency_position_embedding, std=0.02)

    def forward_time_domain(self, x: torch.Tensor) -> torch.Tensor:
        recent = x[:, -self.recent_len:, :]
        tokens = self.field_encoder(recent)
        tokens = tokens + self.time_position_embedding
        local_features = tokens
        for block in self.temporal_blocks:
            local_features = block(local_features)
        global_features = self.time_transformer(tokens)
        gate = self.time_fusion_gate(torch.cat([local_features, global_features], dim=-1))
        fused_tokens = gate * global_features + (1.0 - gate) * local_features
        fused_tokens = self.time_fusion_norm(fused_tokens)
        return self.time_pooling(fused_tokens)

    def forward_frequency_domain(self, x: torch.Tensor) -> torch.Tensor:
        spectrum = torch.fft.rfft(x, dim=1, norm='ortho')
        spectrum = spectrum[:, 1:self.freq_bins + 1, :]
        frequency_features = torch.cat([spectrum.real, spectrum.imag], dim=-1)
        frequency_tokens = self.frequency_projection(frequency_features)
        frequency_tokens = frequency_tokens + self.frequency_position_embedding
        frequency_tokens = self.frequency_transformer(frequency_tokens)
        return self.frequency_pooling(frequency_tokens)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError('输入必须为[B,L,C]')
        if x.shape[1] != self.seq_len:
            raise ValueError(f'序列长度不正确：{x.shape[1]}，要求{self.seq_len}')
        if x.shape[2] != self.input_dim:
            raise ValueError(f'特征数量不正确：{x.shape[2]}，要求{self.input_dim}')
        time_vector = self.forward_time_domain(x)
        frequency_vector = self.forward_frequency_domain(x)
        domain_gate = self.domain_gate(torch.cat([time_vector, frequency_vector], dim=-1))
        fused = domain_gate * time_vector + (1.0 - domain_gate) * frequency_vector
        fused = self.domain_norm(fused)
        return self.head(fused).squeeze(-1)

class FD_CachedFiveDayBuilder:

    def __init__(self, cache_dir, months):
        self.cache_dir = Path(cache_dir)
        self.months = list(months)
        self.month_to_index = {month: index for index, month in enumerate(self.months)}

    def shard_paths(self, month):
        month_index = self.month_to_index[month]
        paths = []
        if month_index > 0:
            paths.append(self.cache_dir / (self.months[month_index - 1] + '.npz'))
        paths.append(self.cache_dir / f'{month}.npz')
        return paths

    def iter_day_batches(self, month):
        parts = [load_cached_shard(path) for path in self.shard_paths(month)]
        x_all = np.concatenate([part['x'] for part in parts], axis=0)
        y_all = np.concatenate([part['y'] for part in parts], axis=0)
        dates_all = np.concatenate([part['dates'] for part in parts], axis=0)
        instruments_all = np.concatenate([part['instruments'] for part in parts], axis=0)
        calendar = np.unique(dates_all)
        calendar.sort()
        date_to_position = {date: index for index, date in enumerate(calendar)}
        row_indices = np.arange(len(dates_all))
        day_maps = {}
        for date in calendar:
            mask = dates_all == date
            mapping = pd.Series(row_indices[mask], index=instruments_all[mask])
            mapping = mapping[~mapping.index.duplicated(keep='last')]
            day_maps[date] = mapping
        target_dates = [date for date in calendar if pd.Timestamp(date).strftime('%Y%m') == month]
        for target_date in target_dates:
            position = date_to_position[target_date]
            if position < 4:
                continue
            required_dates = calendar[position - 4:position + 1]
            common_instruments = day_maps[required_dates[0]].index
            for required_date in required_dates[1:]:
                common_instruments = common_instruments.intersection(day_maps[required_date].index)
            common_instruments = common_instruments.sort_values()
            if len(common_instruments) < MIN_CROSS_SECTION:
                continue
            daily_parts = []
            for required_date in required_dates:
                rows = day_maps[required_date].loc[common_instruments].to_numpy(dtype=np.int64)
                daily_parts.append(x_all[rows])
            x_five_day = np.concatenate(daily_parts, axis=1).astype(np.float32, copy=False)
            target_rows = day_maps[target_date].loc[common_instruments].to_numpy(dtype=np.int64)
            y_raw = y_all[target_rows].astype(np.float32, copy=False)
            valid = np.isfinite(y_raw)
            valid &= np.isfinite(x_five_day).all(axis=(1, 2))
            x_five_day = x_five_day[valid]
            y_raw = y_raw[valid]
            valid_instruments = common_instruments.to_numpy()[valid]
            if len(y_raw) < MIN_CROSS_SECTION:
                continue
            lower, upper = np.quantile(y_raw, [0.01, 0.99])
            y_winsorized = np.clip(y_raw, lower, upper)
            y_centered = y_winsorized - y_winsorized.mean()
            y_std = y_centered.std(ddof=0)
            if not np.isfinite(y_std) or y_std < 1e-08:
                continue
            y_z = y_centered / y_std
            y_z = np.clip(y_z, -LABEL_CLIP, LABEL_CLIP).astype(np.float32)
            y_rank = pd.Series(y_winsorized).rank(method='average', pct=True).to_numpy(dtype=np.float32) * 2.0 - 1.0
            yield {'date': target_date, 'x': x_five_day, 'y_z': y_z, 'y_rank': y_rank, 'y_raw': y_raw, 'instruments': valid_instruments}
        del x_all
        del y_all
        del dates_all
        del instruments_all

def train_and_save(
    datasources,
    model_path=Path(
        "bigalpha_fusion_model.json"
    ),
):
    # 公榜只调用main，不会导入训练脚本。
    # 私榜需要重训时才延迟导入。
    from bigalpha_fusion_train import (
        train_and_save
        as private_train_and_save,
    )

    return private_train_and_save(
        datasources,
        model_path=model_path,
    )

logger = structlog.get_logger()

DATASOURCE_KEY = "bar5m"
INSTRUMENT_TABLE = "bigalpha_2026_instruments"

FUSION_MODEL_PATH = Path(
    "bigalpha_fusion_model.json"
)

ONE_DAY_LEN = 48
FIVE_DAY_LEN = 240

BUFFER_DAYS = 45

V1_WEIGHT = 0.50
V2_WEIGHT = 0.50

V1_BATCH_SIZE = 2048
V2_BATCH_SIZE = 1024

PRICE_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "ask_price1",
    "ask_price2",
    "ask_price3",
    "bid_price1",
    "bid_price2",
    "bid_price3",
]

COUNT_VOLUME_COLUMNS = [
    "deal_number",
    "volume",
    "amount",
    "ask_volume1",
    "ask_volume2",
    "ask_volume3",
    "bid_volume1",
    "bid_volume2",
    "bid_volume3",
    "ask_num_orders1",
    "ask_num_orders2",
    "ask_num_orders3",
    "bid_num_orders1",
    "bid_num_orders2",
    "bid_num_orders3",
]

FEATURE_COLUMNS = (
    PRICE_COLUMNS
    + COUNT_VOLUME_COLUMNS
)


def restore_tensor(meta):
    dtype_name = meta["dtype"]

    if not hasattr(torch, dtype_name):
        raise ValueError(
            f"不支持的张量类型：{dtype_name}"
        )

    return torch.tensor(
        meta["data"],
        dtype=getattr(
            torch,
            dtype_name,
        ),
    ).reshape(
        meta["shape"]
    )


def load_model_payload(
    payload,
    model_class,
    device,
):
    state_dict = {
        name: restore_tensor(meta)
        for name, meta
        in payload[
            "state_dict"
        ].items()
    }

    model = model_class(
        **payload[
            "model_config"
        ]
    ).to(device)

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    feature_columns = list(
        payload[
            "feature_columns"
        ]
    )

    if feature_columns != FEATURE_COLUMNS:
        raise RuntimeError(
            "融合模型的特征顺序不一致"
        )

    feature_mean = np.asarray(
        payload[
            "feature_mean"
        ],
        dtype=np.float32,
    ).reshape(
        1,
        1,
        -1,
    )

    feature_std = np.asarray(
        payload[
            "feature_std"
        ],
        dtype=np.float32,
    ).reshape(
        1,
        1,
        -1,
    )

    feature_std = np.maximum(
        feature_std,
        1e-6,
    ).astype(
        np.float32
    )

    return (
        model,
        feature_mean,
        feature_std,
        payload,
    )


def preprocess_frame(frame):
    frame = frame.copy()

    frame["date"] = pd.to_datetime(
        frame["date"],
        errors="coerce",
    )

    frame["instrument"] = (
        frame["instrument"]
        .astype(str)
    )

    frame = frame.dropna(
        subset=[
            "date",
            "instrument",
        ]
    )

    for column in FEATURE_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    for column in COUNT_VOLUME_COLUMNS:
        frame[column] = np.log1p(
            frame[column].clip(lower=0)
        )

    frame = frame.sort_values(
        [
            "instrument",
            "date",
        ],
        kind="stable",
    ).reset_index(drop=True)

    return frame


def cross_sectional_zscore(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    valid = np.isfinite(values)

    result = np.zeros(
        len(values),
        dtype=np.float64,
    )

    if valid.sum() < 2:
        return result

    valid_values = values[valid]

    mean = float(
        valid_values.mean()
    )

    standard_deviation = float(
        valid_values.std(
            ddof=0
        )
    )

    if (
        not np.isfinite(
            standard_deviation
        )
        or standard_deviation < 1e-12
    ):
        return result

    result[valid] = (
        valid_values - mean
    ) / standard_deviation

    return result


def predict_in_batches(
    model,
    values,
    device,
    batch_size,
):
    outputs = []

    model.eval()

    with torch.inference_mode():
        for start in range(
            0,
            len(values),
            batch_size,
        ):
            batch = torch.from_numpy(
                values[
                    start:
                    start + batch_size
                ]
            ).to(
                device,
                non_blocking=True,
            )

            prediction = model(batch)

            outputs.append(
                prediction
                .detach()
                .cpu()
                .numpy()
            )

            del batch
            del prediction

    if not outputs:
        return np.empty(
            (0,),
            dtype=np.float64,
        )

    return np.concatenate(
        outputs
    ).astype(
        np.float64,
        copy=False,
    )


def query_eligible(
    start_date,
    end_date,
):
    eligible = dai.query(
        f"""
        SELECT
            date,
            instrument
        FROM {INSTRUMENT_TABLE}
        """,
        filters={
            "date": [
                start_date,
                end_date,
            ]
        },
    ).df()

    if eligible.empty:
        return eligible

    eligible["date"] = pd.to_datetime(
        eligible["date"],
        errors="coerce",
    ).dt.normalize()

    eligible["instrument"] = (
        eligible["instrument"]
        .astype(str)
    )

    eligible = (
        eligible
        .dropna(
            subset=[
                "date",
                "instrument",
            ]
        )
        .drop_duplicates(
            [
                "date",
                "instrument",
            ]
        )
        .reset_index(drop=True)
    )

    return eligible


def query_bar_data(
    table,
    query_start,
    query_end,
):
    selected_columns = ", ".join(
        FEATURE_COLUMNS
    )

    frame = dai.query(
        f"""
        SELECT
            date,
            instrument,
            {selected_columns}
        FROM {table}
        ORDER BY instrument, date
        """,
        filters={
            "date": [
                query_start,
                query_end,
            ]
        },
    ).df()

    if frame.empty:
        return frame

    return preprocess_frame(frame)


def build_instrument_day_windows(frame):
    instrument_windows = {}

    all_dates = (
        frame["date"]
        .dt.normalize()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    for instrument, stock_frame in frame.groupby(
        "instrument",
        sort=False,
    ):
        stock_frame = stock_frame.sort_values(
            "date",
            kind="stable",
        ).reset_index(drop=True)

        feature_values = stock_frame[
            FEATURE_COLUMNS
        ].to_numpy(
            dtype=np.float32
        )

        bar_dates = (
            stock_frame["date"]
            .dt.normalize()
            .to_numpy(
                dtype="datetime64[D]"
            )
        )

        close_positions = np.flatnonzero(
            np.append(
                bar_dates[1:]
                != bar_dates[:-1],
                True,
            )
        )

        day_mapping = {}

        for close_position in close_positions:
            if (
                close_position + 1
                < ONE_DAY_LEN
            ):
                continue

            trade_date = pd.Timestamp(
                bar_dates[
                    close_position
                ]
            ).normalize()

            window = feature_values[
                close_position
                - ONE_DAY_LEN
                + 1:
                close_position
                + 1
            ].copy()

            if window.shape != (
                ONE_DAY_LEN,
                len(FEATURE_COLUMNS),
            ):
                continue

            if not np.isfinite(
                window
            ).all():
                continue

            day_mapping[
                trade_date
            ] = window

        if day_mapping:
            instrument_windows[
                str(instrument)
            ] = day_mapping

    return (
        instrument_windows,
        all_dates,
    )


def build_chunk_windows(
    frame,
    target_start,
    target_end,
):
    (
        instrument_windows,
        all_dates,
    ) = build_instrument_day_windows(
        frame
    )

    calendar = [
        pd.Timestamp(date).normalize()
        for date in all_dates
    ]

    date_to_position = {
        date: index
        for index, date
        in enumerate(calendar)
    }

    target_dates = [
        date
        for date in calendar
        if (
            target_start
            <= date
            <= target_end
        )
    ]

    v1_windows = []
    v2_windows = []
    keys = []
    v2_positions = []

    for target_date in target_dates:
        calendar_position = (
            date_to_position[
                target_date
            ]
        )

        required_dates = None

        if calendar_position >= 4:
            required_dates = calendar[
                calendar_position - 4:
                calendar_position + 1
            ]

        for (
            instrument,
            day_mapping,
        ) in instrument_windows.items():
            current_window = (
                day_mapping.get(
                    target_date
                )
            )

            if current_window is None:
                continue

            current_position = len(
                v1_windows
            )

            keys.append(
                (
                    target_date,
                    instrument,
                )
            )

            v1_windows.append(
                current_window
            )

            if required_dates is None:
                continue

            if not all(
                date in day_mapping
                for date in required_dates
            ):
                continue

            five_day_window = np.concatenate(
                [
                    day_mapping[date]
                    for date
                    in required_dates
                ],
                axis=0,
            ).astype(
                np.float32,
                copy=False,
            )

            if five_day_window.shape != (
                FIVE_DAY_LEN,
                len(FEATURE_COLUMNS),
            ):
                continue

            if not np.isfinite(
                five_day_window
            ).all():
                continue

            v2_positions.append(
                current_position
            )

            v2_windows.append(
                five_day_window
            )

    if not v1_windows:
        return (
            np.empty(
                (
                    0,
                    ONE_DAY_LEN,
                    len(FEATURE_COLUMNS),
                ),
                dtype=np.float32,
            ),
            np.empty(
                (
                    0,
                    FIVE_DAY_LEN,
                    len(FEATURE_COLUMNS),
                ),
                dtype=np.float32,
            ),
            np.empty(
                (0,),
                dtype=np.int64,
            ),
            pd.DataFrame(
                columns=[
                    "date",
                    "instrument",
                ]
            ),
        )

    x_v1 = np.stack(
        v1_windows
    ).astype(
        np.float32,
        copy=False,
    )

    if v2_windows:
        x_v2 = np.stack(
            v2_windows
        ).astype(
            np.float32,
            copy=False,
        )
    else:
        x_v2 = np.empty(
            (
                0,
                FIVE_DAY_LEN,
                len(FEATURE_COLUMNS),
            ),
            dtype=np.float32,
        )

    index_frame = pd.DataFrame(
        keys,
        columns=[
            "date",
            "instrument",
        ],
    )

    return (
        x_v1,
        x_v2,
        np.asarray(
            v2_positions,
            dtype=np.int64,
        ),
        index_frame,
    )


def fuse_daily_predictions(frame):
    output_parts = []

    for _, daily in frame.groupby(
        "date",
        sort=True,
    ):
        daily = daily.copy()

        v1_raw = daily[
            "v1_raw"
        ].to_numpy(
            dtype=np.float64
        )

        v2_raw = daily[
            "v2_raw"
        ].to_numpy(
            dtype=np.float64
        )

        common_mask = np.isfinite(
            v2_raw
        )

        score = cross_sectional_zscore(
            v1_raw
        )

        if common_mask.sum() >= 2:
            v1_common_z = (
                cross_sectional_zscore(
                    v1_raw[
                        common_mask
                    ]
                )
            )

            v2_common_z = (
                cross_sectional_zscore(
                    v2_raw[
                        common_mask
                    ]
                )
            )

            score[
                common_mask
            ] = (
                V1_WEIGHT
                * v1_common_z
                + V2_WEIGHT
                * v2_common_z
            )

        # 再对最终融合结果做一次每日横截面标准化。
        daily["score"] = (
            cross_sectional_zscore(
                score
            )
        )

        output_parts.append(
            daily[
                [
                    "date",
                    "instrument",
                    "score",
                ]
            ]
        )

    if not output_parts:
        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "score",
            ]
        )

    return pd.concat(
        output_parts,
        ignore_index=True,
    )


def process_month_chunk(
    table,
    chunk_start,
    chunk_end,
    device,
    v1_model,
    v1_mean,
    v1_std,
    v2_model,
    v2_mean,
    v2_std,
):
    query_start = (
        chunk_start
        - pd.Timedelta(
            days=BUFFER_DAYS
        )
    ).strftime("%Y-%m-%d")

    query_end = (
        chunk_end
        + pd.Timedelta(
            hours=23,
            minutes=59,
            seconds=59,
        )
    ).strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    frame = query_bar_data(
        table=table,
        query_start=query_start,
        query_end=query_end,
    )

    if frame.empty:
        logger.warning(
            "月份行情数据为空",
            start=str(chunk_start),
            end=str(chunk_end),
        )

        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "score",
            ]
        )

    (
        x_v1,
        x_v2,
        v2_positions,
        index_frame,
    ) = build_chunk_windows(
        frame=frame,
        target_start=chunk_start,
        target_end=chunk_end,
    )

    if len(x_v1) == 0:
        logger.warning(
            "月份未生成有效窗口",
            start=str(chunk_start),
            end=str(chunk_end),
        )

        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "score",
            ]
        )

    x_v1 = (
        (
            x_v1
            - v1_mean
        )
        / v1_std
    ).astype(
        np.float32,
        copy=False,
    )

    prediction_v1 = predict_in_batches(
        model=v1_model,
        values=x_v1,
        device=device,
        batch_size=V1_BATCH_SIZE,
    )

    prediction_v2_full = np.full(
        len(index_frame),
        np.nan,
        dtype=np.float64,
    )

    if len(x_v2) > 0:
        x_v2 = (
            (
                x_v2
                - v2_mean
            )
            / v2_std
        ).astype(
            np.float32,
            copy=False,
        )

        prediction_v2 = (
            predict_in_batches(
                model=v2_model,
                values=x_v2,
                device=device,
                batch_size=V2_BATCH_SIZE,
            )
        )

        prediction_v2_full[
            v2_positions
        ] = prediction_v2

    index_frame["v1_raw"] = (
        prediction_v1
    )

    index_frame["v2_raw"] = (
        prediction_v2_full
    )

    fused = fuse_daily_predictions(
        index_frame
    )

    eligible = query_eligible(
        start_date=(
            chunk_start.strftime(
                "%Y-%m-%d"
            )
        ),
        end_date=(
            chunk_end.strftime(
                "%Y-%m-%d 23:59:59"
            )
        ),
    )

    if eligible.empty:
        return fused.iloc[0:0].copy()

    fused["date"] = pd.to_datetime(
        fused["date"]
    ).dt.normalize()

    result = (
        pd.merge(
            fused,
            eligible,
            on=[
                "date",
                "instrument",
            ],
            how="inner",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna(
            subset=["score"]
        )
        .drop_duplicates(
            [
                "date",
                "instrument",
            ]
        )
        [
            [
                "date",
                "instrument",
                "score",
            ]
        ]
        .sort_values(
            [
                "date",
                "instrument",
            ]
        )
        .reset_index(drop=True)
    )

    logger.info(
        "月份融合推理完成",
        start=str(chunk_start.date()),
        end=str(chunk_end.date()),
        rows=len(result),
        days=result[
            "date"
        ].nunique(),
        v2_rows=int(
            np.isfinite(
                prediction_v2_full
            ).sum()
        ),
    )

    return result


def main(
    datasources,
    start_date,
    end_date,
):
    if DATASOURCE_KEY not in datasources:
        raise KeyError(
            "datasources中缺少bar5m，"
            f"实际键为："
            f"{list(datasources.keys())}"
        )

    table = datasources[
        DATASOURCE_KEY
    ]

    start_timestamp = pd.Timestamp(
        start_date
    ).normalize()

    end_timestamp = pd.Timestamp(
        end_date
    ).normalize()

    if end_timestamp < start_timestamp:
        raise ValueError(
            "end_date不能早于start_date"
        )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    if not FUSION_MODEL_PATH.exists():
        raise FileNotFoundError(
            "找不到融合模型文件："
            f"{FUSION_MODEL_PATH}"
        )

    with FUSION_MODEL_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        fusion_payload = json.load(
            file
        )

    if (
        "v1" not in fusion_payload
        or "v2" not in fusion_payload
    ):
        raise RuntimeError(
            "融合JSON缺少v1或v2模型"
        )

    fusion_weights = (
        fusion_payload.get(
            "fusion_weights",
            {}
        )
    )

    if (
        float(
            fusion_weights.get(
                "v1",
                -1.0,
            )
        )
        != V1_WEIGHT
        or float(
            fusion_weights.get(
                "v2",
                -1.0,
            )
        )
        != V2_WEIGHT
    ):
        raise RuntimeError(
            "融合JSON权重不是50%+50%"
        )

    (
        v1_model,
        v1_mean,
        v1_std,
        v1_payload,
    ) = load_model_payload(
        payload=(
            fusion_payload[
                "v1"
            ]
        ),
        model_class=BigAlphaMixer,
        device=device,
    )

    (
        v2_model,
        v2_mean,
        v2_std,
        v2_payload,
    ) = load_model_payload(
        payload=(
            fusion_payload[
                "v2"
            ]
        ),
        model_class=(
            FourierDualDomainBigAlphaMixer
        ),
        device=device,
    )

    logger.info(
        "融合模型加载完成",
        device=str(device),
        v1_model=(
            v1_payload.get(
                "model_name"
            )
        ),
        v2_model=(
            v2_payload.get(
                "model_name"
            )
        ),
        v1_weight=V1_WEIGHT,
        v2_weight=V2_WEIGHT,
    )

    periods = pd.period_range(
        start_timestamp.to_period("M"),
        end_timestamp.to_period("M"),
        freq="M",
    )

    result_parts = []

    for period in periods:
        chunk_start = max(
            start_timestamp,
            period.start_time.normalize(),
        )

        chunk_end = min(
            end_timestamp,
            period.end_time.normalize(),
        )

        chunk_result = (
            process_month_chunk(
                table=table,
                chunk_start=chunk_start,
                chunk_end=chunk_end,
                device=device,
                v1_model=v1_model,
                v1_mean=v1_mean,
                v1_std=v1_std,
                v2_model=v2_model,
                v2_mean=v2_mean,
                v2_std=v2_std,
            )
        )

        if not chunk_result.empty:
            result_parts.append(
                chunk_result
            )

    if not result_parts:
        raise RuntimeError(
            "融合模型没有生成任何有效预测"
        )

    result = (
        pd.concat(
            result_parts,
            ignore_index=True,
        )
        .drop_duplicates(
            [
                "date",
                "instrument",
            ],
            keep="last",
        )
        .sort_values(
            [
                "date",
                "instrument",
            ]
        )
        .reset_index(drop=True)
    )

    result = result[
        [
            "date",
            "instrument",
            "score",
        ]
    ]

    logger.info(
        "融合模型全部推理完成",
        rows=len(result),
        days=result[
            "date"
        ].nunique(),
        instruments=result[
            "instrument"
        ].nunique(),
    )

    return result
